### COPY INTO And MERGE Commands
#### COPY INTO Command
1) Incrementally loads data into Delta Lake tables from Cloud Storage
2) Supports schema evolution
3) Supports wide range of file formats (CSV,JSON, Parquet, Delta)
4) Alternative to Auto Loader for batch ingestion

##### Create the table to copy the data into

In [0]:
CREATE TABLE IF NOT EXISTS demo.delta_lake.raw_stock_prices;

##### Incrementally load new files into the table

In [0]:
DELETE FROM demo.delta_lake.raw_stock_prices;

In [0]:
COPY INTO demo.delta_lake.raw_stock_prices
FROM 'abfss://demo@deawrkspcextdl.dfs.core.windows.net/landing/stock_prices'
FILEFORMAT = JSON
FORMAT_OPTIONS ('inferSchema' = 'true')
COPY_OPTIONS ('mergeSchema' = 'true');

In [0]:
SELECT * FROM demo.delta_lake.raw_stock_prices;

### MERGE Statement
1) Used for upserts (insert/update/delete) operations in a single statement
2) Allows merging new data into a target table based on matching condition

##### Create the target table to merge the data into

In [0]:
CREATE TABLE IF NOT EXISTS demo.delta_lake.stock_prices
(
    stock_id STRING,
    price DOUBLE,
    trading_date DATE
);

###  Merge the source data into target table
- Insert new stocks received
- Update price and trading_date if updates received
- Delete stocks which are de-listed from the exchange (status = 'DELISTED')


In [0]:
MERGE INTO demo.delta_lake.stock_prices AS target
USING demo.delta_lake.raw_stock_prices AS source 
ON target.stock_id = source.stock_id
WHEN MATCHED AND source.status = 'ACTIVE' THEN
    UPDATE SET target.price = source.price, target.trading_date = source.trading_date
WHEN MATCHED AND source.status = 'DELISTED' THEN
    DELETE
WHEN NOT MATCHED AND source.status = 'ACTIVE' THEN
    INSERT (stock_id, price, trading_date) VALUES (source.stock_id, source.price, source.trading_date);

In [0]:
SELECT * FROM demo.delta_lake.stock_prices

In [0]:
SELECT * FROM demo.delta_lake.stock_prices

In [0]:
SELECT * FROM demo.delta_lake.stock_prices